# Notebook 02: Huấn luyện mô hình Seq2Seq trên Google Colab (GPU T4)
**Đề tài NLP-28:** Sửa lỗi ngữ pháp & chuẩn hóa phụ đề ASR tiếng Việt

- Mô hình sử dụng: **vinai/bartpho-syllable** (BART tiếng Việt của VinAI Research, tương thích 100% với Colab).
- Tích hợp Google Drive: Tự động nạp dữ liệu `.txt`/`.srt` từ thư mục `vi-sub-correction/input` và tự động lưu mô hình sau khi huấn luyện vào `vi-sub-correction/checkpoints`.

## Bước 1: Kết nối Google Drive và Tạo thư mục Input / Output

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# 2. Khởi tạo cấu trúc thư mục trên Google Drive của bạn:
# - vi-sub-correction/input: Nơi bạn thả các file .srt hoặc .txt vào
# - vi-sub-correction/checkpoints: Nơi lưu trữ mô hình sau khi train xong
drive_input_dir = Path("/content/drive/MyDrive/vi-sub-correction/input")
drive_ckpt_dir = Path("/content/drive/MyDrive/vi-sub-correction/checkpoints")

drive_input_dir.mkdir(parents=True, exist_ok=True)
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)

print(f"✅ Đã kết nối Google Drive!")
print(f"📁 Thư mục INPUT : {drive_input_dir}")
print(f"📁 Thư mục OUTPUT: {drive_ckpt_dir}")

## Bước 2: Kiểm tra GPU T4 và Cập nhật mã nguồn từ GitHub

In [ ]:
# Kiểm tra phần cứng GPU T4
!nvidia-smi

# Clone hoặc cập nhật bản mới nhất từ GitHub
import os
if not os.path.exists("/content/vi-sub-corretion"):
    !git clone https://github.com/lhhoang5425/vi-sub-corretion.git /content/vi-sub-corretion
else:
    %cd /content/vi-sub-corretion
    !git pull

%cd /content/vi-sub-corretion
!pip install -r requirements.txt -q
print("✅ Môi trường và mã nguồn đã sẵn sàng!")

## Bước 3: Nạp dữ liệu từ Google Drive (SRT / TXT) & Sinh dữ liệu nhiễu
- Hệ thống sẽ tự động quét thư mục `vi-sub-correction/input` trên Google Drive.
- Nếu tìm thấy file `.srt` hoặc `.txt`, code sẽ bóc tách câu và tạo dữ liệu huấn luyện thực tế.
- Nếu chưa có file nào trong thư mục, hệ thống sẽ tự động dùng tập 10 câu mẫu nhân bản để test thử nghiệm pipeline.

In [ ]:
import json
from pathlib import Path
from src.data_processing.noise_generator import ASRNoiseGenerator
from src.srt_utils.srt_processor import srt_to_txt

drive_input_dir = Path("/content/drive/MyDrive/vi-sub-correction/input")
clean_sentences = []

# 1. Đọc tất cả các file .srt và .txt trong thư mục Google Drive
if drive_input_dir.exists():
    for file_path in drive_input_dir.iterdir():
        if file_path.suffix.lower() == ".srt":
            print(f"-> Đang nạp file phụ đề SRT: {file_path.name}")
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                srt_content = f.read()
            extracted_text = srt_to_txt(srt_content)
            lines = [l.strip() for l in extracted_text.split("\n") if len(l.strip()) > 3]
            clean_sentences.extend(lines)
        elif file_path.suffix.lower() == ".txt":
            print(f"-> Đang nạp file văn bản TXT: {file_path.name}")
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    s = line.strip()
                    if len(s) > 3:
                        clean_sentences.append(s)

# 2. Nếu có dữ liệu thật từ Drive thì dùng, nếu chưa có thì dùng câu mẫu
if clean_sentences:
    print(f"✅ Đã nạp thành công {len(clean_sentences)} câu dữ liệu từ Google Drive!")
    seed_sentences = clean_sentences
else:
    print("ℹ️ Chưa thấy file .srt hoặc .txt nào trong thư mục 'vi-sub-correction/input' trên Drive.")
    print("👉 Sử dụng 10 câu mẫu (nhân bản 3,000 dòng) để chạy kiểm thử quy trình train.")
    seed_sentences = [
        "Tôi đi làm ở ngân hàng và dành tiền tiết kiệm hàng tháng.",
        "Hôm nay thời tiết rất đẹp, chúng ta cùng đi dạo phố nhé.",
        "Bác sĩ khuyên bệnh nhân nên nghỉ ngơi và uống thuốc đúng giờ.",
        "Học phần xử lý ngôn ngữ tự nhiên giúp sinh viên hiểu sâu về trí tuệ nhân tạo.",
        "Phim này có phụ đề rất chuẩn xác và âm thanh sống động.",
        "Chúng tôi đang nghiên cứu mô hình ngôn ngữ lớn phục vụ cho doanh nghiệp.",
        "Khách hàng có thể tra cứu thông tin tài khoản qua ứng dụng di động.",
        "Để đạt kết quả học tập tốt, sinh viên cần chủ động làm bài tập lớn.",
        "Dự án này giúp chuẩn hóa phụ đề video một cách tự động và chính xác.",
        "Hệ thống phát hiện gian lận giao dịch ngân hàng hoạt động liên tục ngày đêm."
    ] * 300

# 3. Sinh dữ liệu cặp Noisy -> Clean
generator = ASRNoiseGenerator()
train_data = []
val_data = []

for i, sent in enumerate(seed_sentences):
    noisy, errs = generator.apply_noise(sent)
    record = {"id": i + 1, "noisy": noisy, "clean": sent, "errors": errs}
    if i % 10 == 0:
        val_data.append(record)
    else:
        train_data.append(record)

Path("data/synthetic").mkdir(parents=True, exist_ok=True)
with open("data/synthetic/train.jsonl", "w", encoding="utf-8") as f:
    for r in train_data:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open("data/synthetic/val.jsonl", "w", encoding="utf-8") as f:
    for r in val_data:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"✅ Đã tạo xong: {len(train_data)} mẫu train và {len(val_data)} mẫu validation!")

## Bước 4: Huấn luyện mô hình Seq2Seq (BARTpho - VinAI)

In [ ]:
# Kiểm tra GPU trước khi train
import torch
if not torch.cuda.is_available():
    print('⚠️ CẢNH BÁO: Đang chạy trên CPU! Hãy vào menu Runtime -> Change runtime type -> chọn T4 GPU để train nhanh.')
else:
    print(f'✅ Đã kết nối GPU: {torch.cuda.get_device_name(0)}')

# Chạy huấn luyện với mô hình vinai/bartpho-syllable
!python -m src.models.train_seq2seq \
    --model_name "vinai/bartpho-syllable" \
    --train_file "data/synthetic/train.jsonl" \
    --val_file "data/synthetic/val.jsonl" \
    --output_dir "checkpoints/bartpho-sub-corrector" \
    --epochs 3 \
    --batch_size 8 \
    --lr 5e-5

## Bước 5: Kiểm thử mô hình vừa huấn luyện

In [ ]:
from src.models.inference import Seq2SeqCorrector

# Nạp mô hình vừa train từ checkpoints/bartpho-sub-corrector
my_corrector = Seq2SeqCorrector(model_name_or_path="checkpoints/bartpho-sub-corrector")

test_cases = [
    "toi di lam o ngan hang va gianh tien tiet kiem",
    "bac si bao phai nghi ngoi uong thuoc deu dan",
    "hom nay thoi tiet rat dep chung ta di choi nhe"
]

for text in test_cases:
    cleaned = my_corrector.predict(text)
    print(f"Input : {text}")
    print(f"Output: {cleaned}\n")

## Bước 6: Tự động sao lưu Checkpoint vào Google Drive
- File mô hình sẽ được nén và tự động lưu vào thư mục `vi-sub-correction/checkpoints/` trên Google Drive của bạn.
- Sau này khi mô hình đã học tốt, bạn chỉ cần mở Google Drive trên máy tính để tải file về.

In [ ]:
import shutil
from pathlib import Path

# 1. Nén thư mục checkpoint
print("📦 Đang nén thư mục checkpoint mô hình...")
!zip -r bartpho_checkpoint.zip checkpoints/bartpho-sub-corrector

# 2. Tự động sao chép sang Google Drive
drive_output_zip = "/content/drive/MyDrive/vi-sub-correction/checkpoints/bartpho_checkpoint.zip"
print(f"💾 Đang sao lưu mô hình sang Google Drive: {drive_output_zip} ...")
shutil.copy("bartpho_checkpoint.zip", drive_output_zip)

print("🎉 HOÀN TẤT! Mô hình đã được lưu vĩnh viễn trên Google Drive của bạn:")
print(f"👉 Đường dẫn: {drive_output_zip}")
print("Bạn có thể tải file này về máy tính bất cứ lúc nào từ Google Drive!")